## Problem Formulation

### 1. State Representation

Our state representation is a dictionary with the following structure:

```python
state = {
    'robots': [
        {
            'id': 1,
            'position': (x1, y1),
            'goal': (gx1, gy1),
            'path': [(x1, y1), (x2, y2), ...],
            'path_index': 0,
            'at_goal': False
        },
        {
            'id': 2,
            'position': (x2, y2),
            'goal': (gx2, gy2),
            'path': [(x2, y2), (x3, y3), ...],
            'path_index': 0,
            'at_goal': False
        },
        # ... more robots
    ],
    'time_step': 0,
    'collisions': 0,
    'deadlock': False,
    'grid': GridEnvironment
}
```

**State Variables:**
- `robots[i]['position']`: Current (x, y) position of robot i
- `robots[i]['goal']`: Target (x, y) position for robot i
- `robots[i]['path']`: Planned path as sequence of positions
- `robots[i]['path_index']`: Current progress along path (0 to len(path)-1)
- `robots[i]['at_goal']`: Boolean flag if robot reached goal
- `time_step`: Current simulation time step (0, 1, 2, ...)
- `collisions`: Total number of conflicts detected so far
- `deadlock`: True if robots are stuck (no progress made)
- `grid`: Reference to GridEnvironment object

**Constraints in State:**
- All robot positions must be walkable in grid: `grid.is_walkable(x, y) == True`
- No two robots at same position: `robot_i.position != robot_j.position`
- All positions within grid bounds: `0 <= x < grid.width and 0 <= y < grid.height`


### 2. Actions

**Action Type 1: Plan Paths (Offline Planning)**
```python
action_plan_paths = {
    'type': 'plan_paths',
    'algorithm': 'independent_astar'|'cooperative_astar'|'cbs',
    'robots': [list of robot ids]
}
```
Effect: Each robot computes path from current position to goal
- Independent A*: Each robot plans ignoring others
- Cooperative A*: Robots plan sequentially, avoiding others' paths
- CBS: Hierarchical search resolving conflicts


**Action Type 2: Move Robot (Online Execution)**
```python
action_move = {
    'type': 'move_robot',
    'robot_id': 1,
    'direction': 'up'|'down'|'left'|'right'|'stay'
}
```
Effect: Robot moves one step in direction if valid
- Check: New position is walkable
- Check: No collision with other robot at new position
- Update: `robot.position = (x + dx, y + dy)`
- Update: `time_step += 1`


**Action Type 3: Execute Step (Synchronized Movement)**
```python
action_step = {
    'type': 'execute_step',
    'movements': {
        robot_1_id: (new_x, new_y),
        robot_2_id: (new_x, new_y),
        # ... all robots move simultaneously
    }
}
```
Effect: All robots move one step along their paths
- Precondition: All paths must be planned
- Check: No vertex conflicts (same cell at same time)
- Check: No edge conflicts (crossing paths)
- Update: All `robot.position` values
- Update: All `robot.path_index` incremented
- Update: `time_step += 1`


**Action Type 4: Wait (Robot stays in place)**
```python
action_wait = {
    'type': 'wait',
    'robot_id': 1,
    'steps': 1
}
```
Effect: Robot does not move for specified number of steps
- Used to resolve conflicts via wait-and-retry
- Update: `robot.path_index` stays same, adds wait steps to path


### 3. Goal Test

**Primary Goal**: All robots at their goal positions

```python
def goal_test(state):
    """Check if all robots have reached their goals"""
    for robot in state['robots']:
        if not robot['at_goal']:
            return False
    return True
```

**Goal Conditions:**
- `robot['position'] == robot['goal']` for all robots
- `robot['at_goal'] == True` for all robots
- `time_step` is minimized (makespan criterion)

**Success Criteria:**
1. All robots reach goals ✓
2. No collisions/conflicts ✓
3. No deadlocks ✓
4. Minimum makespan ✓ (secondary objective)
5. Minimum flowtime ✓ (secondary objective)


### 4. Path Cost

**Cost Function (Makespan - Primary Objective):**
```python
def calculate_makespan(state):
    """Time until last robot finishes"""
    return max(len(robot['path']) for robot in state['robots'])
```

**Cost Function (Flowtime - Secondary Objective):**
```python
def calculate_flowtime(state):
    """Sum of all path lengths"""
    total = 0
    for robot in state['robots']:
        total += len(robot['path'])
    return total
```

**Step Cost (Per Move):**
- Each move action costs: 1 unit of time
- Each wait action costs: 1 unit of time

**Penalty Costs:**
- Vertex collision (same cell): +10 cost
- Edge collision (crossing): +10 cost
- Deadlock detected: +100 cost (invalid solution)

**Cost Calculation:**
```python
def path_cost(state):
    """Calculate total cost of current state"""
    # Base cost: time steps taken
    cost = state['time_step']
    
    # Collision penalties
    cost += state['collisions'] * 10
    
    # Deadlock penalty
    if state['deadlock']:
        cost += 100
    
    return cost
```

**Total Evaluation Function:**
```python
def f(state):
    """f(n) = g(n) + h(n)"""
    g_n = path_cost(state)  # Actual cost so far
    h_n = heuristic(state)  # Estimated cost to goal
    return g_n + h_n

def heuristic(state):
    """Admissible heuristic: sum of Manhattan distances"""
    h = 0
    for robot in state['robots']:
        if not robot['at_goal']:
            x1, y1 = robot['position']
            x2, y2 = robot['goal']
            h += abs(x1 - x2) + abs(y1 - y2)
    return h
```


### Initial State

**Initial State Definition:**
```python
initial_state = {
    'robots': [
        {
            'id': 1,
            'position': (1, 1),           # Start position 1
            'goal': (8, 8),               # Goal position 1
            'path': [],                   # Empty (not yet planned)
            'path_index': 0,              # Starting at beginning
            'at_goal': False              # Not at goal yet
        },
        {
            'id': 2,
            'position': (8, 1),           # Start position 2
            'goal': (1, 8),               # Goal position 2
            'path': [],                   # Empty (not yet planned)
            'path_index': 0,              # Starting at beginning
            'at_goal': False              # Not at goal yet
        },
        {
            'id': 3,
            'position': (1, 8),           # Start position 3
            'goal': (8, 1),               # Goal position 3
            'path': [],                   # Empty (not yet planned)
            'path_index': 0,              # Starting at beginning
            'at_goal': False              # Not at goal yet
        },
        # ... more robots
    ],
    'time_step': 0,                       # Time starts at 0
    'collisions': 0,                      # No collisions yet
    'deadlock': False,                    # No deadlock yet
    'grid': GridEnvironment()             # Load grid from file
}
```

**Initial State Example (Concrete):**
```python
# Create grid
grid = GridEnvironment()
grid.load_from_file('warehouse_map.txt')  # 20x20 grid with obstacles

# Create robots
robots = [
    Robot(grid, (1, 1), 'red'),      # Robot 1: Start (1,1)
    Robot(grid, (18, 1), 'blue'),    # Robot 2: Start (18,1)
    Robot(grid, (1, 18), 'green'),   # Robot 3: Start (1,18)
    Robot(grid, (18, 18), 'yellow')  # Robot 4: Start (18,18)
]

# Set goals for each robot
robots[0].set_goal((18, 18))  # Robot 1 goal: (18,18)
robots[1].set_goal((1, 18))   # Robot 2 goal: (1,18)
robots[2].set_goal((18, 1))   # Robot 3 goal: (18,1)
robots[3].set_goal((1, 1))    # Robot 4 goal: (1,1)

# Create initial state
initial_state = {
    'robots': [
        {
            'id': robot.id,
            'position': robot.current_pos,
            'goal': robot.goal_pos,
            'path': [],
            'path_index': 0,
            'at_goal': False
        }
        for robot in robots
    ],
    'time_step': 0,
    'collisions': 0,
    'deadlock': False,
    'grid': grid
}
```

**Properties of Initial State:**
- All robots at distinct start positions
- All start positions are walkable
- No paths planned yet (planning is first action)
- No conflicts exist initially
- Time counter starts at 0
- Cost g(n) = 0 (no moves yet)


### Summary Table

| Component | Description |
|-----------|-------------|
| **State Space** | All possible configurations of n robots on grid |
| **Initial State** | All robots at start positions, no paths planned |
| **Actions** | Plan paths, move robots, wait |
| **Goal Test** | All robots reach goal positions, no conflicts |
| **Path Cost** | Makespan (max robot time) or Flowtime (sum of times) |
| **Constraints** | No overlapping robots, all positions walkable |
| **Search Problem** | Find sequence of moves minimizing makespan |